# 02-1. 정제와 10분 시간축 정렬

02의 1~2장입니다. 결측 코드, 이상값, 중복, 센서 고착을 정리하고 전체 기간의 10분 시간축에 맞춘 뒤 결빙 라벨을 붙입니다.

결과는 `data/processed/`에 저장합니다. `rwis_10min.parquet`(10분 RWIS와 결빙 라벨)은 02-2와 04에서 쓰고, `aws_10min.parquet`, `asos_10min.parquet`와 관측소 정보 3종(`*_stations.parquet`)은 02-2에서 씁니다.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]  # notebooks/<단계 폴더>/ 에서 실행
sys.path.insert(0, str(ROOT))

from src import config

In [2]:
import glob
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. 정제

결측 코드, 이상값, 중복을 자료별로 처리합니다. 주최 측이 안내한 결측값(-99.9, -99.2, -99.7, -999, 빈값, QnA 2026-10-07)은 여기서 뺍니다. 예외는 ASOS 적설·일조의 빈칸입니다. 두 열은 눈이 없거나 밤이면 비어 있어서(01-2의 3-2) 장애로 보지 않고 0으로 둡니다.

| 자료 | 결측 표기 | 이상값 기준 |
|------|-----------|-------------|
| RWIS | -999 | 노면상태: 설명자료에 없는 0 / 풍속 60 m/s 초과 / 기온·노면온도 물리 범위 밖 / 노면온도가 6시간 이상 같은 값(센서 고착, 노면상태도 함께 제거) |
| AWS | -99.x (-99 이하) | 기온·습도·풍속 범위 밖 |
| ASOS | 빈칸 | 적설·일조는 빈칸을 0(현상 없음)으로 봄 |

In [3]:
def load_observations(source):
    """일별 CSV를 기간 전체로 합쳐서 불러온다."""
    files = sorted(glob.glob(str(config.RAW_DIR / config.RAW_FILES[source])))
    return pd.concat((pd.read_csv(f, encoding=config.CSV_ENCODING) for f in files), ignore_index=True)


def clip_range(df, ranges):
    """범위를 벗어난 값을 NaN으로 바꾼다."""
    for col, (lo, hi) in ranges.items():
        df[col] = df[col].where(df[col].between(lo, hi))
    return df


RWIS_COLS = {"노면상태": "state", "노면온도 (광학)": "rst", "기온 (C)": "ta", "습도 (%)": "rh",
             "1분 풍속 (m/s)": "ws", "일 누적 강수량 (mm)": "pr_day", "시정 (m)": "vis"}
VALID_STATES = [1, 2, 3, 4, 10, 11, 12, 13, 14, 15]


def clean_rwis(raw):
    df = raw.rename(columns=RWIS_COLS)
    df["time"] = pd.to_datetime(df["관측시각 (KST)"].astype(str), format="%Y%m%d%H%M")
    df["stn"] = df["관측 지점번호"]
    df = df[["stn", "time", *RWIS_COLS.values()]].replace(-999, np.nan)
    df["state"] = df["state"].where(df["state"].isin(VALID_STATES))
    df = clip_range(df, {"rst": (-30, 60), "ta": (-30, 40), "rh": (1, 100), "ws": (0, 60),
                         "pr_day": (0, 300), "vis": (1, 50000)})
    df = df.drop_duplicates(["stn", "time"]).sort_values(["stn", "time"]).reset_index(drop=True)
    return mask_stuck(df, "rst", ["rst", "state"])


STUCK_STEPS = 36  # 6시간


def mask_stuck(df, col, cols_to_mask, steps=STUCK_STEPS):
    """같은 값이 steps번(10분 단위) 이상 이어지는 구간을 센서 고착으로 보고 결측으로 바꾼다 (01-2의 3-4 참고)."""
    same = df[col].eq(df.groupby("stn")[col].shift()) & df[col].notna()
    run_len = df.groupby((~same).cumsum())[col].transform("size")
    stuck = run_len >= steps
    if stuck.any():
        print(f"센서 고착으로 결측 처리: {int(stuck.sum())}행", df.loc[stuck, "stn"].value_counts().to_dict(),
              f"({df.loc[stuck, 'time'].min()} ~ {df.loc[stuck, 'time'].max()})")
    df.loc[stuck, cols_to_mask] = np.nan
    return df


raw_rwis = load_observations("rwis")
rwis = clean_rwis(raw_rwis)
print(rwis.shape)
rwis.describe().T.round(2)

센서 고착으로 결측 처리: 3897행 {23202: 3897} (2025-11-19 10:30:00 ~ 2025-12-16 11:50:00)
(587445, 9)


,count,mean,min,25%,50%,75%,max,std
stn,587445.0,22936.601185,12103.0,12409.0,23204.0,33206.0,33412.0,8853.49748
time,587445,2025-07-01 11:22:46.107806,2024-11-01 00:00:00,2024-12-30 23:50:00,2025-02-28 23:40:00,2025-12-30 23:30:00,2026-02-28 23:50:00,NaN
state,583358.0,3.033676,1.0,1.0,1.0,2.0,15.0,4.506057
rst,583520.0,4.294702,-13.3,-0.7,3.5,8.6,33.1,6.753424
ta,310986.0,2.983475,-17.4,-1.4,2.4,7.0,24.4,6.294332
rh,310986.0,71.333345,6.9,58.1,73.6,86.9,100.0,18.855899
ws,310817.0,1.313118,0.0,0.6,1.1,1.7,11.2,0.947547
pr_day,310986.0,0.289941,0.0,0.0,0.0,0.0,22.0,1.325477
vis,309789.0,23526.080384,10.0,11630.0,22560.0,35364.0,50000.0,13862.002104


In [4]:
# 관측소 정보 (지점별로 값이 하나뿐인 것 확인)
assert raw_rwis.groupby("관측 지점번호")[["관측소 위도", "관측소 경도"]].nunique().max().max() == 1
stations = (
    raw_rwis.groupby("관측 지점번호")
    .agg(name=("관측 지점명", "first"), route=("고속도로", "first"), kind=("관측소 종류", "first"),
         lat=("관측소 위도", "first"), lon=("관측소 경도", "first"), alt_given=("관측소 고도 (m)", "first"))
    .rename_axis("stn").reset_index()
)
for c in ["name", "route", "kind"]:
    stations[c] = stations[c].str.strip()
stations["holdout"] = stations["stn"].isin(config.HOLDOUT_STATIONS)
stations

,stn,name,route,kind,lat,lon,alt_given,holdout
0,12103,군산휴(도),서해안,거점,36.0202,126.7959,33,False
1,12209,서김제(도),서해안,기본,35.8347,126.8398,12,False
2,12210,부안고려청자휴(도),서해안,기본,35.6733,126.7345,10,False
3,12211,고인돌휴(도),서해안,기본,35.4643,126.6730,52,False
4,12409,우곡3교(도),서해안,목표(결빙),35.9853,126.8249,13,False
5,12410,부곡교(도),서해안,목표(결빙),35.6433,126.7141,17,True
6,23202,여산휴(도),호남선,기본,36.0476,127.1036,70,False
7,23203,삼례졸음(도),호남선,기본,35.9185,127.0710,25,False
8,23204,정읍휴(도),호남선,기본,35.6002,126.8626,31,False
9,23402,해전(도),호남선,목표(결빙),35.8999,127.0578,13,True


In [5]:
AWS_COLS = {"1분 평균 기온(°C)": "ta", "이슬점온도(°C)": "td", "1분 평균 상대습도(%)": "rh",
            "10분 평균 풍속(m/s)": "ws", "강수감지(0-무강수)": "rain", "60분 누적 강수량(mm)": "pr60",
            "12시간 누적 강수량(mm)": "pr12h", "일 누적 강수량(mm)": "pr_day"}


def clean_aws(raw):
    df = raw.rename(columns=AWS_COLS)
    df["time"] = pd.to_datetime(df["시간(KST)"])
    df["stn"] = df["지점 번호"]
    cols = list(AWS_COLS.values())
    df[cols] = df[cols].mask(df[cols] <= -99)  # -99.9, -99.7, -99.2 등 결측 코드
    df = clip_range(df, {"ta": (-35, 40), "td": (-50, 35), "rh": (1, 100), "ws": (0, 60), "rain": (0, 1),
                         "pr60": (0, 200), "pr12h": (0, 500), "pr_day": (0, 500)})
    return df[["stn", "time", *cols]].drop_duplicates(["stn", "time"])


raw_aws = load_observations("aws")
aws = clean_aws(raw_aws)
aws_st = (raw_aws.groupby("지점 번호").agg(name=("지점명", "first"), lat=("위도", "first"), lon=("경도", "first"))
          .rename_axis("stn").reset_index())
print(aws.shape, "결측 비율:", aws.isna().mean().round(3).to_dict())

(1152144, 10) 결측 비율: {'stn': 0.0, 'time': 0.0, 'ta': 0.003, 'td': 0.004, 'rh': 0.004, 'ws': 0.007, 'rain': 0.013, 'pr60': 0.015, 'pr12h': 0.023, 'pr_day': 0.014}


In [6]:
ASOS_COLS = {"전운량(10분위)": "cloud", "지면온도(°C)": "tg", "적설(cm)": "snow", "일조(hr)": "sunshine"}


def clean_asos(raw):
    df = raw.rename(columns=ASOS_COLS)
    df["time"] = pd.to_datetime(df["시간"])
    df["stn"] = df["지점"]
    df[["snow", "sunshine"]] = df[["snow", "sunshine"]].fillna(0)  # 빈칸 = 현상 없음(눈 없음, 야간)
    df = clip_range(df, {"cloud": (0, 10), "tg": (-30, 60), "snow": (0, 200), "sunshine": (0, 1)})
    return df[["stn", "time", *ASOS_COLS.values()]].drop_duplicates(["stn", "time"])


raw_asos = load_observations("asos")
asos = clean_asos(raw_asos)
asos_st = (raw_asos.groupby("지점").agg(name=("지점명", "first"), lat=("위도", "first"), lon=("경도", "first"))
           .rename_axis("stn").reset_index())
print(asos.shape, "결측 비율:", asos.isna().mean().round(3).to_dict())

(57600, 6) 결측 비율: {'stn': 0.0, 'time': 0.0, 'cloud': 0.002, 'tg': 0.005, 'snow': 0.0, 'sunshine': 0.0}


목표안개 관측소(1-1)는 시정만 2개소에서 재서 이번 분석에 쓰지 않았습니다. ASOS 일사는 5개소에만 일사계가 있어서 모든 지점에 있는 일조를 대신 씁니다.

## 2. 10분 시간축 정렬 및 결빙 라벨

분석 기간(config.PERIODS) 전체의 10분 시간축을 만들고 지점마다 빠진 시각은 결측으로 채웁니다. ASOS(1시간)는 10분 시간축에 맞춰 선형 보간합니다(최대 50분). 두 겨울 사이(2025-03 ~ 2025-10)는 비어 있어서 보간과 이동 통계는 기간별로 따로 계산합니다.

In [7]:
TIMES = pd.DatetimeIndex(np.concatenate([
    pd.date_range(start, pd.Timestamp(end) + pd.Timedelta("23:50:00"), freq=config.TIME_STEP)
    for start, end in config.PERIODS
]))
PERIOD_ID = np.concatenate([np.full(len(pd.date_range(s, pd.Timestamp(e) + pd.Timedelta("23:50:00"),
                                                      freq=config.TIME_STEP)), i)
                            for i, (s, e) in enumerate(config.PERIODS)])
PERIOD_SLICES = [np.flatnonzero(PERIOD_ID == i) for i in range(len(config.PERIODS))]
print(f"10분 시간축 {len(TIMES):,}개: {TIMES[0]} ~ {TIMES[-1]}")


def to_wide(df, col, stns):
    """긴 형식(stn, time, 값)을 (지점 x 시각) 배열로 바꾼다. 없는 시각은 NaN."""
    wide = df.pivot(index="time", columns="stn", values=col).reindex(index=TIMES, columns=stns)
    return wide.to_numpy(dtype="float32").T


def per_period(frame, func):
    """시각 x 지점 DataFrame에 func를 기간별로 적용해 다시 붙인다."""
    return pd.concat([func(frame.iloc[idx]) for idx in PERIOD_SLICES])


def hourly_to_10min(df, col, stns):
    """1시간 자료를 10분 시간축으로 선형 보간한다 (기간별, 최대 5칸)."""
    wide = df.pivot(index="time", columns="stn", values=col).reindex(columns=stns)
    wide = wide.reindex(wide.index.union(TIMES)).loc[TIMES]
    wide = per_period(wide, lambda f: f.interpolate(limit=5, limit_area="inside").ffill(limit=5))
    return wide.to_numpy(dtype="float32").T


RWIS_STNS = stations["stn"].tolist()
idx = pd.MultiIndex.from_product([RWIS_STNS, TIMES], names=["stn", "time"])
rwis = rwis.set_index(["stn", "time"]).reindex(idx).reset_index()
rwis["ice"] = rwis["state"].isin(config.ICE_STATE_CODES).astype("float32").where(rwis["state"].notna())
print("정렬 후 RWIS:", rwis.shape)
print("노면상태 결측(라벨 없음):", rwis["state"].isna().sum(), "/ Ice:", int(rwis["ice"].sum()))

10분 시간축 34,560개: 2024-11-01 00:00:00 ~ 2026-02-28 23:50:00


정렬 후 RWIS: (587520, 10)
노면상태 결측(라벨 없음): 4162 / Ice: 303


In [8]:
AWS_STNS = aws_st["stn"].tolist()
ASOS_STNS = asos_st["stn"].tolist()
AWS = {v: to_wide(aws, v, AWS_STNS) for v in AWS_COLS.values()}
ASOS = {v: hourly_to_10min(asos, v, ASOS_STNS) for v in ASOS_COLS.values()}
print("AWS 배열:", AWS["ta"].shape, "ASOS 배열:", ASOS["cloud"].shape)
print("ASOS 10분 보간 후 결측 비율:", {k: round(float(np.isnan(v).mean()), 3) for k, v in ASOS.items()})

AWS 배열: (34, 34560) ASOS 배열: (10, 34560)
ASOS 10분 보간 후 결측 비율: {'cloud': 0.002, 'tg': 0.005, 'snow': 0.0, 'sunshine': 0.0}


## 저장

10분 시간축에 맞춘 RWIS(결빙 라벨 포함), AWS, ASOS와 관측소 정보를 저장합니다. AWS·ASOS는 (지점 x 시각) 배열을 긴 형식(stn, time, 변수)으로 바꿔 저장합니다.

In [9]:
def to_long(arrays, stns):
    """(지점 x 시각) 배열 묶음을 긴 형식(stn, time, 변수...)으로 바꾼다."""
    out = pd.DataFrame({"stn": np.repeat(stns, len(TIMES)), "time": np.tile(TIMES, len(stns))})
    for var, a in arrays.items():
        out[var] = a.ravel()
    return out


out = config.PROCESSED_DIR
rwis.to_parquet(out / "rwis_10min.parquet", index=False)
stations.to_parquet(out / "rwis_stations.parquet", index=False)
aws_st.to_parquet(out / "aws_stations.parquet", index=False)
asos_st.to_parquet(out / "asos_stations.parquet", index=False)
to_long(AWS, AWS_STNS).to_parquet(out / "aws_10min.parquet", index=False)
to_long(ASOS, ASOS_STNS).to_parquet(out / "asos_10min.parquet", index=False)
for name in ["rwis_10min", "rwis_stations", "aws_stations", "asos_stations", "aws_10min", "asos_10min"]:
    print(f"{name + '.parquet':24s} {(out / f'{name}.parquet').stat().st_size / 1e6:8.1f} MB")

rwis_10min.parquet            3.9 MB
rwis_stations.parquet         0.0 MB
aws_stations.parquet          0.0 MB
asos_stations.parquet         0.0 MB
aws_10min.parquet             8.2 MB
asos_10min.parquet            1.6 MB
